![Week 2 Day 5](../assets/banners/w2d5.png)

# Week 2 Day 5 - 🧱 Project Day: Measuring Retrieval, and the GradeWise Ingestion Pipeline

What a week! On Day 1 you turned words into vectors; on Day 2 you loaded and chunked the PDFs; on Day 3 they went into Chroma with idempotent ingestion; and yesterday you searched them with dense, BM25 and hybrid retrieval and built a RAG chain with citations.

But here's the uncomfortable question Asha will ask before she lets GradeWise anywhere near her students' marks: **"How do you know the search finds the right notes?"** "It looked fine on three questions" isn't an answer. Today we turn "seems to work" into numbers: MRR and nDCG over 60 test questions, broken down by category. Then we walk through the project code, from the *simple* pipeline to the *pro* one, and wrap up the week.

![Your 6-week roadmap](../assets/roadmap/week2.png)

### 🎯 By the end of this lab you will
1. 🧮 Compute reciprocal rank, MRR and nDCG by hand, and check them against GradeWise's functions
2. 📏 Run `week2/evaluation/eval.py` on 60 questions and read the per-category table
3. ⚖️ Compare the simple and pro ingestion pipelines fairly, and know exactly what each pro choice buys
4. 🧱 Know the week's milestone commands: ingest, answer, evaluate

### 🎬 Videos for this lab

| # | Lecture |
|:-:|---|
| 1 | 🧱 Day 5 - Measuring Retrieval: MRR, nDCG and Recall@k with `tests.jsonl` |
| 2 | 🧱 Day 5 - From Simple to Pro: The GradeWise Ingestion Pipeline and CLI |
| 3 | Day 5 - Week 2 Wrap-Up: Summary, Roadmap and Quiz |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Today's evaluation runs <b>offline</b> with <code>--offline --pipeline bm25</code>. Because BM25 doesn't use embeddings, <b>those numbers are real</b>, not a plumbing check. The <code>--offline</code> flag also puts its vector store in a separate folder (<code>output/offline_eval/</code>), so fake vectors never mix with your real knowledge base. Dense and hybrid scores need real embeddings: with a key, drop <code>--offline</code> (a fraction of a cent).</span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ OpenAI API Key not set. That's OK for most of this lab: the offline cells use fakes.")
    print("   The few cells that call a real model will tell you so. Setup: Part 5 of the setup guide.")

In [ ]:
import importlib.util
import json
import math
import subprocess
import sys
from collections import Counter

from langchain_core.documents import Document

from gradewise.config import REPO_ROOT


def load_module(path, name):
    """Import a .py file by path (our week folders aren't packages)."""
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


evaluation = load_module(REPO_ROOT / "week2" / "evaluation" / "eval.py", "week2_eval")


def run(*args: str) -> None:
    """Run a course script with this notebook's Python, from the repo root, and show its output."""
    env = {**os.environ, "PYTHONIOENCODING": "utf-8", "COLUMNS": "100"}
    result = subprocess.run(
        [sys.executable, *args], cwd=REPO_ROOT, capture_output=True, text=True, encoding="utf-8", env=env
    )
    print(result.stdout, result.stderr, sep="")


print("✅ eval.py loaded from", REPO_ROOT / "week2" / "evaluation")

## Part 1: The test set, `tests.jsonl`

Open `week2/evaluation/tests.jsonl`. Each line is one question a CS201 student (or a grader looking for evidence) might ask, with:
- `keywords`: phrases that a relevant chunk **must** contain
- `reference_answer`: what a careful marker would accept (used by the optional LLM judge)
- `source`: the PDF the answer comes from
- `category`: `direct_fact`, `numerical`, `comparison`, `spanning` (answer spread over several chunks) or `holistic` (needs the big picture)

`week2/evaluation/test.py` loads every line with Pydantic, so a typo fails loudly instead of silently skewing the scores.

In [ ]:
tests = evaluation.load_tests()
print(f"{len(tests)} test questions\n")
print("By category:", dict(Counter(t.category for t in tests)))
print("By source  :", dict(Counter(t.source for t in tests)), "\n")
print(json.dumps(tests[3].model_dump(), indent=2, ensure_ascii=False))

## Part 2: The metrics, by hand

Suppose the question is *"What are the four necessary conditions for deadlock?"*, the keyword is **"circular wait"**, and a retriever returns these five chunks:

In [ ]:
retrieved = [
    Document("Learning outcomes: define deadlock and state the four conditions."),
    Document("Mutual exclusion and hold and wait are two of the conditions..."),
    Document("**Circular**\nwait: a cycle of processes, each waiting for the next."),
    Document("Deadlock vs starvation: starvation is waiting indefinitely..."),
    Document("Breaking circular wait: order the resources and acquire them in order."),
]
relevant = [evaluation._contains(d, "circular wait") for d in retrieved]
print("Contains 'circular wait'?", relevant)

💡 Notice chunk 3 says `**Circular**\nwait`, with Markdown bold and a line break in the middle. The evaluator normalises both away before matching, otherwise pymupdf4llm's formatting would hide real hits.

### Reciprocal rank and MRR

**Reciprocal rank (RR)** = 1 / (rank of the *first* relevant chunk). First relevant at rank 3 → RR = 1/3. If nothing relevant comes back, RR = 0. **MRR** is the mean of the RRs: in `eval.py`, over each question's keywords, then over all questions.

In [ ]:
first_hit = relevant.index(True) + 1
print(f"By hand    : first hit at rank {first_hit} → RR = 1/{first_hit} = {1 / first_hit:.3f}")
print(f"eval.py    : {evaluation.calculate_mrr('circular wait', retrieved):.3f}")

# MRR over several keywords = the mean of their reciprocal ranks
keywords = ["circular wait", "hold and wait", "starvation"]
rrs = [evaluation.calculate_mrr(kw, retrieved) for kw in keywords]
print(f"\nRRs for {keywords} = {[round(r, 3) for r in rrs]} → MRR = {sum(rrs) / len(rrs):.3f}")

### nDCG: are *all* the relevant chunks near the top?

MRR only cares about the first hit. But an answer that spans several chunks needs *all* of them, and preferably near the top, where the LLM pays most attention. **DCG@k** adds up each relevant chunk, discounted by its position:

$$\text{DCG@}k = \sum_{i=1}^{k} \frac{\text{rel}_i}{\log_2(i + 1)}$$

Rank 1 counts fully, rank 2 counts 0.63, rank 3 counts 0.5, rank 5 only 0.39. **nDCG** divides by the DCG of the *ideal* ordering of the same chunks, so it always lands between 0 and 1.

In [ ]:
rel = [1.0 if r else 0.0 for r in relevant]
for i, r in enumerate(rel, start=1):
    discount = 1 / math.log2(i + 1)
    print(f"rank {i}: rel = {r:.0f}, discount 1/log2({i + 1}) = {discount:.3f} → adds {r * discount:.3f}")

dcg = sum(r / math.log2(i + 1) for i, r in enumerate(rel, start=1))
ideal = sum(r / math.log2(i + 1) for i, r in enumerate(sorted(rel, reverse=True), start=1))
print(f"\nDCG = {dcg:.3f}, ideal DCG = {ideal:.3f} → nDCG = {dcg / ideal:.3f}")
print(f"eval.py: {evaluation.calculate_ndcg('circular wait', retrieved, k=5):.3f}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Two retrievers each find the same two relevant chunks in their top 3. Retriever A returns them at ranks 1 and 2; retriever B at ranks 2 and 3. Both have Keywords@3 = 100%. Predict: what are their MRRs, and their nDCG@3s? Which metric tells them apart more strongly?</span>
        </td>
    </tr>
</table>

In [ ]:
def as_docs(*flags: int) -> list[Document]:
    return [Document("tcp" if f else "nothing") for f in flags]


for name, flags in (("A", (1, 1, 0)), ("B", (0, 1, 1))):
    docs = as_docs(*flags)
    mrr = evaluation.calculate_mrr("tcp", docs)
    ndcg = evaluation.calculate_ndcg("tcp", docs, k=3)
    print(f"Retriever {name} {flags}: MRR = {mrr:.3f}, nDCG@3 = {ndcg:.3f}")

Both metrics prefer A. MRR halves for B because its *first* hit moved down one place; nDCG drops less, because B still found both chunks, just a little lower. That's why we report both.

The other two numbers `eval.py` prints are simpler:
- **Keywords** (coverage, a form of *recall@k*): the share of a question's keywords found *anywhere* in the top k
- **Source@k**: did *any* chunk from the expected PDF make the top k?

## Part 3: Run the evaluation

Time for the real thing: all 60 questions, the BM25 pipeline, k = 5. This is the same command you'd type in a terminal: `uv run week2/evaluation/eval.py --offline --pipeline bm25`.

In [ ]:
run("week2/evaluation/eval.py", "--offline", "--pipeline", "bm25")

### Reading the per-category table

Look at the rows, not just the `ALL` line. That's where the insight is:

- **`comparison`, `numerical` and `direct_fact`** score high. These questions use the notes' own vocabulary ("compare TCP and UDP", "/26 subnet", "PCB"), which is BM25's home ground.
- **`spanning`** drops: the answer lives in several chunks (sometimes several sections), and the keywords from the second half often don't make the top 5.
- **`holistic`** is the weakest: "summarise the databases unit" needs the big picture, and no single 1,000-character chunk *is* the big picture.
- **Source@5** is high almost everywhere: we nearly always reach the right PDF, but not always the right *paragraph*. That's the gap between "roughly relevant" and "useful evidence".

Let's look inside one of the hard ones, row 55 (a holistic question), with `--row`:

In [ ]:
run("week2/evaluation/eval.py", "--offline", "--pipeline", "bm25", "--row", "55")

Each line shows a retrieved chunk's citation and which of the expected keywords it contains. "From a browser to a web server" touches DNS (2.6), the handshake (2.5), encapsulation (2.2) and HTTPS (2.6): four sections, five slots, and BM25 spends some slots on chunks that just repeat the question's words.

How would you fix it? Bigger chunks? A larger k? Multi-query (one rewrite per layer)? Hybrid search with real embeddings? **That's exactly what this week's exercise asks you to measure.**

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">A metric is only as good as its ground truth. Ours is <b>keyword-based</b>: a chunk "counts" if it contains the exact phrase. That makes it cheap, free and repeatable, but it has blind spots: a chunk that paraphrases the answer scores 0, and bigger chunks contain more keywords just by being bigger. Always sanity-check a surprising score with <code>--row</code>, and for answer quality add <code>--answers --limit 10</code> (an LLM judge, a few cents).</span>
        </td>
    </tr>
</table>

## Part 4: From simple to pro

The week's project code lives in three folders. Open them side by side:

| | `week2/implementation/` (simple) | `week2/pro_implementation/` (pro) |
|---|---|---|
| Loader | `pymupdf` raw text | pymupdf4llm Markdown (`gradewise.ingestion.loaders`) |
| Cleaning | none: running headers and "Page X of Y" in every chunk | headers, footers, page numbers stripped; ligatures normalised |
| Chunking | fixed 500 characters, 200 overlap | structure-aware: `##` sections first, 1,000 / 150, contextual header |
| Metadata | `source`, `page` | + `title`, `unit`, `section`, `section_path`, `chunk_index`, `file_hash` |
| Re-runs | wipes and re-embeds everything, every time | idempotent: unchanged files skipped, $0 |
| Retrieval | dense only | hybrid (dense + BM25, RRF), with metadata filters |
| Prompt | "use the notes" | cite every fact; "don't cover that" rather than guess; notes are data, not instructions |

`week2/evaluation/eval.py --compare` runs both pipelines on all 60 questions and prints "simple → pro" in every cell. Offline, though, that comparison is meaningless, because the simple pipeline is *dense-only* and its embeddings would be fake.

So let's do a comparison that **is** meaningful offline: keep the retriever fixed (BM25) and change only the **ingestion**. Same 60 questions, same metrics; the only difference is how the PDFs were loaded and chunked.

In [ ]:
import tempfile

from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise.config import DATA_DIR
from gradewise.ingestion import chunk_documents, get_vectorstore, load_directory
from gradewise.rag import get_retriever

simple = load_module(REPO_ROOT / "week2" / "implementation" / "ingest.py", "week2_simple_ingest")
simple_chunks = simple.create_chunks(simple.fetch_documents())  # raw pymupdf, fixed 500/200
pro_chunks = chunk_documents(load_directory(DATA_DIR / "knowledge-base"))  # the pro choices
print(f"simple: {len(simple_chunks)} chunks · pro: {len(pro_chunks)} chunks")

# ⚠️ Offline: fake embeddings in a temp folder. BM25 ignores them, so the scores below are real.
TMP = tempfile.mkdtemp(prefix="gradewise_w2d5_")


def bm25_scores(name: str, chunks: list[Document], k: int = 5) -> dict:
    store = get_vectorstore(name, embeddings=DeterministicFakeEmbedding(size=32), persist_dir=TMP)
    for i, chunk in enumerate(chunks):
        chunk.metadata.setdefault("chunk_index", i)  # 💡 the simple chunks have no index; BM25 needs an identity
    store.add_documents(chunks, ids=[f"{name}-{i}" for i in range(len(chunks))])
    retriever = get_retriever(store=store, k=k, mode="bm25")
    results = [evaluation.evaluate_retrieval(t, retriever.invoke(t.question), k) for t in tests]
    return evaluation.summarize(tests, results)


summaries = {"simple ingest": bm25_scores("simple", simple_chunks), "pro ingest": bm25_scores("pro", pro_chunks)}
evaluation.print_summary(summaries, k=5)

Same retriever, same questions: the ingestion choices alone move the scores. The pro version's chunks follow the sections, carry a contextual header that names their topic, and are free of ligatures and running headers, so BM25 finds the right paragraph more often.

With a key, the full comparison including dense and hybrid retrieval is one command (it builds whichever store is missing first):

```bash
uv run week2/evaluation/eval.py --compare
```

### The ingestion pipeline and the CLI

The pro ingestion script is tiny, because the work lives in the package. Open `week2/pro_implementation/ingest.py`: it calls `gradewise.ingestion.ingest()`, which you ran on Day 3 with fake embeddings. The same command is also part of GradeWise's CLI:

In [ ]:
run("-m", "gradewise", "ingest", "--help")

Now build your **real** knowledge base, with `gradewise.models.embeddings()`. It embeds about 70 chunks, which costs a few hundredths of a cent with `text-embedding-3-small`. Without a key you'll see GradeWise's friendly error panel instead: nothing breaks.

In [ ]:
run("week2/pro_implementation/ingest.py")

Run that cell a second time (with a key): "4 unchanged", **0 chunks added**, $0. Then ask the pro pipeline a question from the command line:

```bash
uv run week2/pro_implementation/answer.py "What are the four necessary conditions for deadlock?"
uv run week2/evaluation/eval.py                        # the pro pipeline, real embeddings
uv run week2/evaluation/eval.py --answers --limit 10   # plus an LLM judge on 10 answers (a few cents)
```

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">An evaluation set is the most valuable artefact in an AI project, and the one teams most often skip. It turns every debate ("should we use bigger chunks?", "is the expensive embedding model worth it?", "did that refactor break anything?") into a two-minute experiment with a number at the end. Sixty questions written by someone who knows the material, as here, is enough to catch most regressions. Start one on day one of any RAG project.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">This week's milestone: <b>the CS201 knowledge base is ingested and its retrieval quality is measured.</b> The files: <code>gradewise/ingestion/</code> (loaders, chunking, store, pipeline), <code>gradewise/rag/</code> (retrievers, citations, answer), and <code>week2/evaluation/</code> (tests.jsonl, test.py, eval.py). In week 4, GradeWise's grader will call this same retriever as a tool to find evidence for every mark it awards.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

### 🗓️ Week 2 in one table

| Day | You learned | It lives in |
|:-:|---|---|
| 1 | Why RAG; embeddings and cosine similarity | `gradewise/models.py` (`embeddings()`) |
| 2 | pymupdf4llm loading, header stripping, structure-aware chunking | `gradewise/ingestion/loaders.py`, `chunking.py` |
| 3 | Chroma, collections, metadata filters, idempotent ingestion | `gradewise/ingestion/store.py`, `pipeline.py` |
| 4 | Dense, BM25, hybrid (RRF), MMR, multi-query, reranking, cited answers | `gradewise/rag/retrievers.py`, `citations.py`, `answer.py` |
| 5 | MRR, nDCG, coverage and Source@k; simple vs pro | `week2/evaluation/`, `week2/implementation/`, `week2/pro_implementation/` |

🧱 **Milestone reached:** course PDFs → chunked, embedded, searchable knowledge base, with retrieval quality *measured*.

📝 **Quiz 2** (10 questions) is on Udemy at the end of the section. 🏋️ **This week's exercise:** `week2/week2_EXERCISE.ipynb`: tune the chunking and beat the baseline nDCG.

**Next week (Week 3): Agents & LangGraph.** So far every pipeline has been a straight line. Next week GradeWise learns to *decide*: we'll rebuild exam understanding as a LangGraph graph with state, retries when validation fails, and a human-in-the-loop pause where Asha approves the extracted exam before anything gets graded.

⏭ **Next:** `week2/week2_EXERCISE.ipynb`, then `week3/day1.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Two warm-ups before the week exercise. (1) Re-run the evaluation with <code>--k 3</code> and <code>--k 10</code>. Which metrics change most, and why can Keywords only go up as k grows? (2) Use <code>--row</code> on a <code>spanning</code> question (try 58 or 59) and write one sentence on why it scores low. Then open <code>week2_EXERCISE.ipynb</code> and try to beat the baseline nDCG. Solutions are in <code>week2/solutions/day5_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• Ed Donner's RAG evaluation pattern (the inspiration for <code>tests.jsonl</code>): <a href="https://github.com/ed-donner/llm_engineering">github.com/ed-donner/llm_engineering</a><br/> • nDCG explained: <a href="https://en.wikipedia.org/wiki/Discounted_cumulative_gain">en.wikipedia.org/wiki/Discounted_cumulative_gain</a><br/> • LangChain retrieval evaluation with LangSmith: <a href="https://docs.langchain.com/langsmith/evaluate-rag-tutorial">docs.langchain.com/langsmith/evaluate-rag-tutorial</a></span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/thankyou.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">Thank you</h2>
            <span style="color:#7c5cff;">That's week 2 done, and GradeWise now knows the course. Thank you for sticking with the details: loaders, chunk boundaries and metrics aren't glamorous, but they're what separates a demo from something Asha can trust. See you in week 3, where GradeWise starts making decisions of its own.</span>
        </td>
    </tr>
</table>